In [ ]:


import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

# ============================================================
# CONFIG
# ============================================================
TRAIN_DIR = "/kaggle/input/datasets/sabbir4724/training-data"   # <-- set your training image dataset path
VAL_DIR   = "/kaggle/input/datasets/sabbir4724/testing-data"     # <-- set your validation image dataset path

NUM_CLASSES = 4            # expected number of clusters for evaluation (k-means k)
IMAGE_SIZE = 64             # standard VAE works best on smaller resolutions
LATENT_DIM = 128
BATCH_SIZE = 64
NUM_EPOCHS = 5
LR = 1e-3
KL_WEIGHT = 1.0             # beta in the ELBO (1.0 = standard VAE, not beta-VAE)
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

# ============================================================
# DATA
# ============================================================
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),  # pixels in [0, 1], matches Bernoulli/BCE decoder output
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=4, pin_memory=True, drop_last=True,
                           persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=4, pin_memory=True)

print(f"Train images: {len(train_dataset)} | Val images: {len(val_dataset)}")
print(f"Val classes (for evaluation only): {val_dataset.classes}")


# ============================================================
# MODEL: standard convolutional VAE
# (encoder -> mu, logvar -> reparameterize -> decoder)
# ============================================================
class Encoder(nn.Module):
    def __init__(self, latent_dim, image_size):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),   # -> 32 x H/2
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # -> 64 x H/4
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # -> 128 x H/8
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),# -> 256 x H/16
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.feat_size = image_size // 16
        flat_dim = 256 * self.feat_size * self.feat_size
        self.fc_mu = nn.Linear(flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(flat_dim, latent_dim)

    def forward(self, x):
        h = self.conv(x)
        h = torch.flatten(h, 1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, latent_dim, image_size):
        super().__init__()
        self.feat_size = image_size // 16
        flat_dim = 256 * self.feat_size * self.feat_size
        self.fc = nn.Linear(latent_dim, flat_dim)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),
            nn.Sigmoid(),  # output in [0, 1]
        )

    def forward(self, z):
        h = self.fc(z)
        h = h.view(-1, 256, self.feat_size, self.feat_size)
        recon = self.deconv(h)
        return recon


class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, image_size=IMAGE_SIZE):
        super().__init__()
        self.encoder = Encoder(latent_dim, image_size)
        self.decoder = Decoder(latent_dim, image_size)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

    @torch.no_grad()
    def encode_mean(self, x):
        mu, _ = self.encoder(x)
        return mu


model = VAE().to(DEVICE)


# ============================================================
# LOSS: ELBO = reconstruction (BCE) + KL divergence
# ============================================================
def vae_loss(recon, x, mu, logvar, kl_weight=KL_WEIGHT):
    recon_loss = F.binary_cross_entropy(recon, x, reduction="sum") / x.size(0)
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    total = recon_loss + kl_weight * kl_div
    return total, recon_loss, kl_div


# ============================================================
# TRAINING
# ============================================================
def train_one_epoch(net, loader, optimizer, epoch):
    net.train()
    total_loss, total_recon, total_kl, total_n = 0.0, 0.0, 0.0, 0

    pbar = tqdm(loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}", leave=True)
    for images, _ in pbar:
        images = images.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        recon, mu, logvar = net(images)
        loss, recon_loss, kl_div = vae_loss(recon, images, mu, logvar)
        loss.backward()
        optimizer.step()

        bs = images.size(0)
        total_loss += loss.item() * bs
        total_recon += recon_loss.item() * bs
        total_kl += kl_div.item() * bs
        total_n += bs

        pbar.set_postfix(loss=f"{loss.item():.2f}",
                          recon=f"{recon_loss.item():.2f}",
                          kl=f"{kl_div.item():.2f}")

    return total_loss / total_n, total_recon / total_n, total_kl / total_n


# ============================================================
# EVALUATION on the separate validation dataset
# ============================================================
@torch.no_grad()
def extract_val_latents(net):
    net.eval()
    latents = []
    for images, _ in val_loader:
        images = images.to(DEVICE)
        mu = net.encode_mean(images)
        latents.append(mu.cpu().numpy())
    return np.concatenate(latents, axis=0)


def hungarian_match(pred_clusters, true_labels, num_clusters, num_classes):
    size = max(num_clusters, num_classes)
    cm = np.zeros((size, size), dtype=np.int64)
    for p, t in zip(pred_clusters, true_labels):
        cm[p, t] += 1
    row_ind, col_ind = linear_sum_assignment(-cm)
    mapping = {r: c for r, c in zip(row_ind, col_ind)}
    return np.array([mapping[p] for p in pred_clusters])


def evaluate_on_val(net):
    val_latents = extract_val_latents(net)
    val_true_labels = np.array(val_dataset.targets)
    num_classes = len(val_dataset.classes)

    km = KMeans(n_clusters=NUM_CLASSES, n_init=20, random_state=SEED)
    val_clusters = km.fit_predict(val_latents)

    mapped_preds = hungarian_match(val_clusters, val_true_labels, NUM_CLASSES, num_classes)

    acc = accuracy_score(val_true_labels, mapped_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        val_true_labels, mapped_preds, average="macro", zero_division=0
    )
    return acc, precision, recall, f1


# ============================================================
# MAIN LOOP
# ============================================================
def main():
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(NUM_EPOCHS):
        t0 = time.time()
        loss, recon, kl = train_one_epoch(model, train_loader, optimizer, epoch)
        dt = time.time() - t0
        print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
              f"loss={loss:.4f} recon={recon:.4f} kl={kl:.4f} time={dt:.1f}s")

    acc, precision, recall, f1 = evaluate_on_val(model)
    print("\n=== Validation Results (k-means on latent means, Hungarian-matched vs true labels) ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    torch.save(model.state_dict(), "vae_model.pth")
    print("\nModel saved to vae_model.pth")


if __name__ == "__main__":
    main()

In [ ]:
"""
Benchmark: Params (M), Time/Image (ms), FPS
For the VAE model (encoder + decoder) built in vae_raw.py.

Loads the trained weights if available (vae_model.pth), otherwise benchmarks
the freshly initialized architecture (architecture/size is identical either
way — only weight values differ, which don't affect speed or param count).
"""

import time
import torch
import torch.nn as nn

# ============================================================
# CONFIG (must match vae_raw.py)
# ============================================================
IMAGE_SIZE = 64
LATENT_DIM = 128
CHECKPOINT_PATH = "vae_model.pth"   # set to None to skip loading weights

NUM_WARMUP = 20      # warmup iterations (excluded from timing, lets GPU/cuDNN settle)
NUM_RUNS = 200        # timed iterations
BATCH_SIZE = 1         # 1 = true per-image latency; increase to benchmark throughput

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# MODEL (identical architecture to vae_raw.py)
# ============================================================
class Encoder(nn.Module):
    def __init__(self, latent_dim, image_size):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.feat_size = image_size // 16
        flat_dim = 256 * self.feat_size * self.feat_size
        self.fc_mu = nn.Linear(flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(flat_dim, latent_dim)

    def forward(self, x):
        h = self.conv(x)
        h = torch.flatten(h, 1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, latent_dim, image_size):
        super().__init__()
        self.feat_size = image_size // 16
        flat_dim = 256 * self.feat_size * self.feat_size
        self.fc = nn.Linear(latent_dim, flat_dim)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z):
        h = self.fc(z)
        h = h.view(-1, 256, self.feat_size, self.feat_size)
        recon = self.deconv(h)
        return recon


class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, image_size=IMAGE_SIZE):
        super().__init__()
        self.encoder = Encoder(latent_dim, image_size)
        self.decoder = Decoder(latent_dim, image_size)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar


# ============================================================
# LOAD MODEL
# ============================================================
model = VAE(LATENT_DIM, IMAGE_SIZE).to(DEVICE)

if CHECKPOINT_PATH:
    try:
        state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
        model.load_state_dict(state_dict)
        print(f"Loaded weights from {CHECKPOINT_PATH}")
    except FileNotFoundError:
        print(f"'{CHECKPOINT_PATH}' not found — benchmarking untrained architecture "
              f"(params/speed are identical to the trained model).")

model.eval()


# ============================================================
# PARAMS (M)
# ============================================================
def count_params(net):
    total = sum(p.numel() for p in net.parameters())
    trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
    return total, trainable


total_params, trainable_params = count_params(model)
params_m = total_params / 1e6


# ============================================================
# TIME / IMAGE (ms) + FPS
# ============================================================
@torch.no_grad()
def benchmark(net, image_size, batch_size, num_warmup, num_runs, device):
    dummy_input = torch.randn(batch_size, 3, image_size, image_size, device=device)

    # Warmup (lets cuDNN autotuner / GPU clocks settle — excluded from timing)
    for _ in range(num_warmup):
        _ = net(dummy_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    timings = []
    for _ in range(num_runs):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        _ = net(dummy_input)

        if device.type == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        timings.append((t1 - t0) * 1000.0)  # ms for this batch

    timings = torch.tensor(timings)
    mean_batch_ms = timings.mean().item()
    std_batch_ms = timings.std().item()

    time_per_image_ms = mean_batch_ms / batch_size
    fps = 1000.0 / time_per_image_ms

    return time_per_image_ms, fps, mean_batch_ms, std_batch_ms


time_per_image_ms, fps, mean_batch_ms, std_batch_ms = benchmark(
    model, IMAGE_SIZE, BATCH_SIZE, NUM_WARMUP, NUM_RUNS, DEVICE
)


# ============================================================
# REPORT
# ============================================================
print("\n=== VAE Benchmark ===")
print(f"Device            : {DEVICE}")
print(f"Image size         : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size (timed) : {BATCH_SIZE}")
print(f"Total Params (M)   : {params_m:.3f}")
print(f"Trainable Params (M): {trainable_params / 1e6:.3f}")
print(f"Batch latency (ms) : {mean_batch_ms:.3f} ± {std_batch_ms:.3f}")
print(f"Time/Image (ms)    : {time_per_image_ms:.3f}")
print(f"FPS                : {fps:.2f}")

: 